# Remote Server Tutorial

> **Note:** Code cells in this notebook contain shell commands to run in a terminal — they are not meant to be executed as notebook cells. Commands labeled **\[local\]** should be run on your own machine; commands labeled **\[server\]** should be run after SSH-ing into the server.

---

## 0. Prerequisites

Before you start, make sure you have the following on your local machine:

- **CMU VPN** — you must be connected to the VPN to reach the server. Install it here if needed:  
  https://www.cmu.edu/computing/services/endpoint/network-access/vpn/how-to/index.html
- **VS Code** with the **Remote - SSH** extension:
  1. Open the Extensions sidebar (`Cmd+Shift+X` / `Ctrl+Shift+X`)
  2. Search for **Remote - SSH** (publisher: Microsoft, extension ID `ms-vscode-remote.remote-ssh`)
  3. Click Install
- **Git** installed
- **Python 3** installed
- Your Andrew ID and password

---
## 1. Initial Connection

Connect to the server using your Andrew ID and password. This first login uses password authentication — we'll switch to SSH key authentication in the next section.

**\[local\]**

In [ ]:
ssh YOUR_ANDREW_ID@heinz-90803.heinz.cmu.edu

After entering your password, the prompt will change to something like:

```
YOUR_ANDREW_ID@heinz-90803:~$
```

You are now on the server. Commands you type here run on the remote machine, not on your laptop.

If you don't trust me, you can run the following command to see where you are:


In [ ]:
hostname

Now, run the following command for the next step:

In [ ]:
mkdir -p ~/.ssh

When you're ready to move on to SSH key setup, type `exit` to disconnect.

---
## 2. SSH Key Setup

If you connect to a server regularly, it can be cumbersome to enter your credentials every time. SSH keys let you log in without typing a password. You have a key **pair**: a public key (`.pub`) that you can share freely, and a private key (no extension) that you must never share with anyone.

### 2.1 Check for an Existing Key

**\[local\]** — run this to see if you already have an SSH key pair:

In [ ]:
ls ~/.ssh/*.pub

If you see `id_ed25519.pub` or `id_rsa.pub`, you already have a key pair — skip to Section 2.2.

**If no key exists**, generate one **\[local\]**:

In [ ]:
ssh-keygen -t ed25519 -C "YOUR_ANDREW_ID@andrew.cmu.edu"

Press Enter to accept the default save path (`~/.ssh/id_ed25519`). You can optionally add a passphrase, although I wouldn't recommend it. This creates two files:

- `~/.ssh/id_ed25519` — your **private** key. Never share this file.
- `~/.ssh/id_ed25519.pub` — your **public** key. Safe to share.

### 2.2 Copy Your Public Key to the Server with `scp`

`scp` (secure copy) transfers files over SSH using the same credentials and host addressing. Its syntax mirrors `cp`, but uses `user@host:path` for remote locations:

```
scp <source> <destination>
```

Copy your public key from your local machine to the server **\[local\]**:

In [ ]:
scp ~/.ssh/id_ed25519.pub YOUR_ANDREW_ID@heinz-90803.heinz.cmu.edu:~/.ssh/authorized_keys

**Note:** What you just did was send a copy of your public key (okay to share) to the server and saved it as a special file called `authorized_keys`, which tells the server which public keys you've authorized. Typically, if you already had an `authorized_keys` file with other public keys in it, you would need to send your key first and then **append** it to that file using something like, `cat ~/.ssh/id_ed25519.pub >> ~/.ssh/authorized_keys` on the server. We skip that step here for efficiency, but worth noting that `authorized_keys` is a special type of file, not just a randomly chosen name for a single key.


Now, try to connect to the server again and see if it asks you for a password. Note: if you set up your ssh key with a passphrase, it will still ask you to enter that. Can't say I didn't warn you!

In [ ]:
ssh YOUR_ANDREW_ID@heinz-90803.heinz.cmu.edu

Did it work?

`exit` (ctrl-D) from the server one more time.

### 2.3 SSH Config: Give the Server a Nickname

Okay, one more tip: Typing `YOUR_ANDREW_ID@heinz-90803.heinz.cmu.edu` every time is tedious, and when you're working on multiple servers it becomes unmanageable fast. An SSH config file maps short aliases to full connection details.

Open your config file in a text editor **\[local\]**:

In [ ]:
nano ~/.ssh/config

Add this block, replacing `YOUR_ANDREW_ID`:

```
Host mlfp
    HostName heinz-90803.heinz.cmu.edu
    User YOUR_ANDREW_ID
    IdentityFile ~/.ssh/id_ed25519
    ServerAliveInterval 60
    ServerAliveCountMax 5
```

What each field does:
- `Host mlfp` — "mlfp" (or whatever you choose) will be an alias for the longer server address (i.e., `YOUR_ANDREW_ID@heinz-90803.heinz.cmu.edu`)
- `HostName` — the real server address
- `User` — your login name, so you don't need to type `andrew_id@` each time
- `IdentityFile` — which private key to use for authentication
- `ServerAliveInterval` / `ServerAliveCountMax` — sends keepalive pings so idle connections aren't dropped

In `nano`, save with `Ctrl+O` then Enter, and exit with `Ctrl+X`.

Now connect with just **\[local\]**:

In [ ]:
ssh mlfp

The alias works with `scp` too:

```bash
scp somefile.csv mlfp:~/some/path/
```

You can add as many different servers as you want to your ssh config just like you did above, which makes it easier to login and manage different settings you might want to use for them.

### 2.4 Connect with VS Code Remote-SSH

With the Remote - SSH extension installed:

1. Open the Command Palette (`Cmd+Shift+P` / `Ctrl+Shift+P`)
2. Type **Remote-SSH: Connect to Host** and select it
3. Select `mlfp` from the list (or type `YOUR_ANDREW_ID@heinz-90803.heinz.cmu.edu`)
4. A new VS Code window opens — the file explorer and integrated terminal are now on the server
5. Use **File → Open Folder** to open a directory on the server

Inside this window, saving a file saves it directly to the server. Take a second to think about that. The server only offers a command line interface, but with VS-Code, you can write code on the server as easily as if you were doing it on your own laptop. Pretty cool.

---
## 3. Add Your Server's SSH Key to GitHub

To push code from the server to GitHub, GitHub needs to authenticate the server. The cleanest way is SSH: you generate a key pair on the server, then register the public key in your GitHub account. This is separate from the key you use to log in to the server — each machine that communicates with GitHub needs its own entry.

**\[server\]** — generate an SSH key pair on the server to use for github:

In [ ]:
ssh-keygen -t ed25519 -C "YOUR_ANDREW_ID@andrew.cmu.edu"

Accept the default path and skip the passphrase (press Enter twice). Now print the public key:

In [ ]:
cat ~/.ssh/id_ed25519.pub

Copy the entire output (it starts with `ssh-ed25519`). Then add it to your GitHub account:

1. Go to **GitHub → Settings → SSH and GPG keys**
2. Click **New SSH key**
3. Give it a descriptive title like `mlfp server`
4. Paste the public key into the Key field
5. Click **Add SSH key**

Test that GitHub recognizes the server **\[server\]**:

In [ ]:
ssh -T git@github.com

You should see: `Hi YOUR_GITHUB_USERNAME! You've successfully authenticated...`

If prompted about the host's authenticity, type `yes` to continue.

---
## 4. Fork and Clone the Repo

First, **fork** the repo on GitHub so you have your own copy to push to:

1. Go to the repo on GitHub
2. Click **Fork** (top-right)
3. Accept the defaults and click **Create fork**

Then clone **your fork** on the server **\[server\]**:

In [ ]:
git clone git@github.com:YOUR_GITHUB_USERNAME/remote_server_tutorial_demo.git
cd remote_server_tutorial_demo

---
## 5. Set Up a Virtual Environment

The first thing to do after cloning is create a Python virtual environment. On a shared server, everyone uses the same system Python, so installing packages globally creates version conflicts between users. A virtual environment is isolated to your project directory.

> **Note:** If you ever need to install a Python package outside a venv on this server, add `--user` to the `pip` command (e.g., `pip install somepackage --user`). Inside a venv this flag is not needed.

**\[server\]**

In [ ]:
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt

When the environment is active, your prompt shows `(.venv)` at the start. To deactivate:

```bash
deactivate
```

Remember to run `source .venv/bin/activate` again any time you open a new terminal session on the server.

---
## 6. Running Python Code

There are two ways to run Python code on the server.

### 6.1 Command Line

Make sure your venv is active, then run **\[server\]**:

In [ ]:
python src/hello_remote.py

### 6.2 Jupyter Notebook in VS Code

With VS Code Remote-SSH you can open `.ipynb` notebook files directly — the notebook runs on the server with no SSH tunnel needed.

Before opening a notebook, register your virtual environment as a Jupyter kernel so VS Code can use it **\[server\]**:

In [ ]:
python -m ipykernel install --user --name .venv --display-name "Python (.venv)"

Open `remote_demo.ipynb` in VS Code. When prompted to select a kernel, choose **Python (.venv)**.

Run the cells one by one. The last cell imports `tqdm` — you should get a `ModuleNotFoundError` since it isn't installed in your venv yet. Try installing it yourself and re-run the cell.

---
## Helpful Reference Commands

### Navigation **\[server\]**

In [ ]:
pwd           # print current directory
ls -lah       # list files with sizes and permissions
cd ~          # go to home directory
cd ..         # go up one level
cd -          # go to previous directory
mkdir mydir   # create a directory
cp a b        # copy file a to b
mv a b        # move or rename file a to b
rm file       # delete a file
cat file      # print file contents
head file     # first 10 lines
tail -f file  # follow a file as it grows (useful for logs)

### Monitor Resource Usage

In [ ]:
top    # see the running jobs on the server, type 'q' to quit

### Redirect Output to a Log File

In [ ]:
python <some_script_with_output>.py > job.log 2>&1  # redirect stdout and stderr to a file
tail -f job.log                     # watch the output as it arrives

### Text Editors

In [ ]:
nano file.txt  # beginner-friendly: Ctrl+O to save, Ctrl+X to exit
vim file.txt   # powerful, steeper learning curve: i to insert, Esc then :wq to save and quit

### Good Practices on a Shared Server

- Do not leave unnecessary jobs running
- Check CPU and memory usage before launching heavy jobs (`top` or `htop`)
- Do not edit, move, or delete other users' files
- Keep your work organized inside your own home directory
- Save large shared datasets/models to a common location to avoid saving the same things in multiple locations